# UWSM Community Survey — Analysis Notebook

**Project:** Listening to Southern Maine: A Data-Driven Look at Community Challenges and Civic Engagement  
**Author:** Hangliang Ren ([@myhott163com](https://github.com/myhott163com))  
**Partner:** United Way of Southern Maine  
**Last revised:** 2026-05

This notebook runs the descriptive statistics, statistical tests, ML models, and visualizations behind the project report. It assumes `data/processed_data_5.csv` already exists (output of `notebooks/01_cleaning_processing.ipynb`). If you don't have the cleaned file, run notebook 01 first to generate it from the raw Excel.

The notebook is organized into eight sections:

1. **Setup** — imports and style configuration
2. **Load and filter** — read the processed CSV, restrict to Southern Maine
3. **Descriptive statistics** — sample composition, top challenges, demographics
4. **Key visualizations** — every figure that appears in the report and deck
5. **Statistical validation** — chi-square tests with Cramér's V across three relationship families
6. **Predictive modeling** — logistic regression and decision tree for Below-ALICE classification
7. **Dimensionality reduction** — PCA on the 24-feature challenge matrix
8. **Wrap-up** — pointers to the report for interpretation

Every figure-generating cell saves a PNG into the working directory and also displays inline.


## 1. Setup

Imports and the UWSM colour palette / style configuration used by every figure in the project. The palette deliberately matches United Way of Southern Maine's brand colours so the figures drop into the community-facing deck without rework.

In [ ]:
# Core data stack
import numpy as np
import pandas as pd
from collections import Counter

# Statistics
from scipy.stats import chi2_contingency

# Machine learning
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from wordcloud import WordCloud

# Reproducibility — every model and clusterer in this notebook uses this seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# ── UWSM brand palette ───────────────────────────────────────────────────────
# These colours appear in every figure; matching the UWSM deck means figures
# generated here can drop directly into community-facing presentations.
BLUE_ACCENT  = "#003DA5"   # primary
RED_ACCENT   = "#C8102E"   # secondary
AMBER_ACCENT = "#E87722"   # tertiary

# Neutral palette for cards, labels, backgrounds
CARD_FILL    = "#FFFFFF"
LABEL_COLOR  = "#4A4A4A"
SUB_COLOR    = "#888780"
BG_COLOR     = "#F0F2F5"

# Candara is the deck's display font. If unavailable, matplotlib will fall
# back to a default sans-serif — non-fatal.
plt.rcParams["font.family"] = "Candara"


## 2. Load and filter

Read the processed dataset and restrict the analytic sample to Southern Maine (Cumberland County, York County, or both). Respondents coded as `Outside Southern Maine` are excluded from every analysis below.


In [ ]:
df = pd.read_csv("../data/processed_data_5.csv")
print(f"Loaded {len(df):,} respondents")
print(f"Columns: {len(df.columns)}")


In [ ]:
# Restrict to Southern Maine analytic sample
df_sm = df[df["live_work_sm_clean"] != "Outside Southern Maine"].copy()
print(f"Southern Maine analytic sample: {len(df_sm):,} respondents")
print(f"  Cumberland only:     {(df_sm['live_work_sm_clean'] == 'Cumberland').sum():,}")
print(f"  York only:           {(df_sm['live_work_sm_clean'] == 'York').sum():,}")
print(f"  Cumberland & York:   {(df_sm['live_work_sm_clean'] == 'Cumberland & York').sum():,}")


## 3. Descriptive statistics

Sample composition, top-line challenge frequencies, and per-instrument coverage. All numbers here use the Southern Maine analytic sample (`df_sm`).

### 3.1 ALICE distribution

In [ ]:
# ALICE: 'Above ALICE' (could cover $400 emergency) vs 'Below ALICE' (could not / maybe)
alice_counts = df_sm["alice_clean"].value_counts()
alice_pct    = df_sm["alice_clean"].value_counts(normalize=True) * 100

print("ALICE distribution (Southern Maine sample):")
for status in alice_counts.index:
    print(f"  {status}: {alice_counts[status]:,}  ({alice_pct[status]:.1f}%)")


### 3.2 Top challenges overall

In [ ]:
# Challenge columns are one-hot indicators derived from the multi-select biggest_challenge field
challenge_cols = [c for c in df_sm.columns if c.startswith("biggest_challenge_")
                  and "clean" not in c]

# Citation counts across the Southern Maine sample
challenge_counts = df_sm[challenge_cols].apply(pd.to_numeric, errors="coerce").sum()
challenge_counts = challenge_counts.sort_values(ascending=False)

print("Top 15 challenges (Southern Maine):")
for label, n in challenge_counts.head(15).items():
    clean_label = label.replace("biggest_challenge_", "").replace("_", " ")
    print(f"  {clean_label:<40} {int(n):>5,}")


### 3.3 Civic engagement: feel heard and want involvement

In [ ]:
# Feel heard — collapsed to binary in cleaning notebook
print("── Feel heard (Southern Maine, respondents who answered) ──")
print(df_sm["feel_heard_clean"].value_counts())
print()

# Want involvement
print("── Want to get involved ──")
print(df_sm["want_involvement_clean"].value_counts())
print()

# How — multi-select preferences
involvement_cols = [c for c in df_sm.columns if c.startswith("how_involvement_")
                    and "clean" not in c]
print("── How they want to get involved (counts, all respondents) ──")
inv_counts = df_sm[involvement_cols].apply(pd.to_numeric, errors="coerce").sum().sort_values(ascending=False)
for label, n in inv_counts.items():
    clean_label = label.replace("how_involvement_", "").replace("_", " ")
    print(f"  {clean_label:<20} {int(n):>4}")


### 3.4 Demographics — age and race/ethnicity

Both fields are only asked on the Full Survey and ALICE Voices instruments. Counts here are restricted to respondents who were asked the question.

In [ ]:
# Age group
print("── Age group (Full Survey + ALICE Voices only) ──")
print(df_sm["age_group_clean"].value_counts())
print()

# Race/ethnicity — multi-select, also restricted to instruments that asked
race_cols = [c for c in df_sm.columns if c.startswith("race_ethnicity_")
             and "clean" not in c]
race_counts = df_sm[race_cols].apply(pd.to_numeric, errors="coerce").sum().sort_values(ascending=False)
print("── Race/ethnicity ──")
for label, n in race_counts.items():
    clean_label = label.replace("race_ethnicity_", "").replace("_", " ")
    print(f"  {clean_label:<25} {int(n):>4}")


### 3.5 Community write-in: which communities feel unheard

In [ ]:
# The community field is open-text classified through NER + zero-shot pipeline
# (see notebook 01). community_super holds the super-category collapse used in the figure.
community_cols = [c for c in df_sm.columns if c.startswith("community_")
                  and "clean" not in c and "fine" not in c and "super" not in c]
comm_counts = df_sm[community_cols].apply(pd.to_numeric, errors="coerce").sum().sort_values(ascending=False)

print("── Identified communities (Southern Maine, classified write-ins) ──")
for label, n in comm_counts.items():
    clean_label = label.replace("community_", "").replace("_", " ")
    print(f"  {clean_label:<25} {int(n):>4}")


## 4. Key visualizations

Every figure used in the report and the deck. Each cell saves a `.png` into the current working directory and displays inline. Filenames match the ones already committed in `figures/` so re-running the notebook overwrites them in place.


### 4.1 Summary metrics — hero card layout

Five-card summary used as the opening slide. The card layout is built manually with `FancyBboxPatch` rather than a chart library because the visual style matches UWSM's deck templates.

In [ ]:
# ── COMPUTE STATS ─────────────────────────────────────────────────────────────
total_n = len(df_sm)

region_pcts         = df_sm["live_work_sm_clean"].value_counts(normalize=True) * 100
cumberland_pct      = region_pcts.get("Cumberland", 0)
york_pct            = region_pcts.get("York", 0)
cumberland_and_york = region_pcts.get("Cumberland & York", 0)

alice_pcts      = df_sm["alice_clean"].value_counts(normalize=True) * 100
below_alice_pct = alice_pcts.get("Below ALICE", 0)

heard_pcts  = df_sm["feel_heard_clean"].value_counts(normalize=True) * 100
unheard_pct = heard_pcts.get("Unheard", 0)

involve_pcts      = df_sm["want_involvement_clean"].value_counts(normalize=True) * 100
want_involved_pct = involve_pcts.get("Yes", 0)

# ── CARD DATA ─────────────────────────────────────────────────────────────────
row1 = [
    (f"{total_n:,}",              "TOTAL RESPONDENTS",       "Cumberland & York Counties",   BLUE_ACCENT),
    (f"{cumberland_pct:.0f}% / {york_pct:.0f}%",
                                  "CUMBERLAND / YORK",       f"{cumberland_and_york:.0f}% listed both",
                                                                                              RED_ACCENT),
    (f"{below_alice_pct:.0f}%",   "BELOW ALICE THRESHOLD",   "Struggling to afford basics",  AMBER_ACCENT),
]
row2 = [
    (f"{unheard_pct:.0f}%",       "FEEL THEIR VOICE IS UNHEARD", "Answered \'Yes\' or \'Maybe\'", RED_ACCENT),
    (f"{want_involved_pct:.0f}%", "WANT TO GET INVOLVED",        "Ready to be part of the solution", BLUE_ACCENT),
]


def draw_card(ax, x, y, w, h, number, label, sublabel, accent):
    """Render one stat card: rounded white background with a coloured accent bar on the left."""
    # Rounded white card
    card = mpatches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.07",
        linewidth=0, facecolor=CARD_FILL, zorder=2)
    ax.add_patch(card)

    # Accent bar — drawn at double width so right corners hide behind a second white card
    bar_w = 0.13
    bar = mpatches.FancyBboxPatch(
        (x, y), bar_w * 2, h,
        boxstyle="round,pad=0.07",
        linewidth=0, facecolor=accent, zorder=3)
    ax.add_patch(bar)
    card2 = mpatches.FancyBboxPatch(
        (x + bar_w, y), w - bar_w, h,
        boxstyle="round,pad=0.0",
        linewidth=0, facecolor=CARD_FILL, zorder=4)
    ax.add_patch(card2)

    cx = x + bar_w + (w - bar_w) / 2
    ax.text(cx, y + h * 0.72, number,    ha="center", va="center",
            fontsize=22, fontweight="bold", color=accent, zorder=5)
    ax.text(cx, y + h * 0.44, label,     ha="center", va="center",
            fontsize=10, fontweight="bold", color=LABEL_COLOR, zorder=5)
    ax.text(cx, y + h * 0.18, sublabel,  ha="center", va="center",
            fontsize=9.5, color=SUB_COLOR, zorder=5)


# ── FIGURE ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")

ax.text(5, 5.72, "About the Data: Southern Maine Community Survey",
        ha="center", va="center", fontsize=15, fontweight="bold", color=LABEL_COLOR)
ax.text(5, 5.42, "Respondents from Cumberland and York Counties  |  Community Survey + ALICE Voices",
        ha="center", va="center", fontsize=9.5, color=SUB_COLOR, style="italic")

card_w, card_h, gap = 2.8, 1.9, 0.3
total_w1 = 3 * card_w + 2 * gap
start_x1 = (10 - total_w1) / 2
for i, (number, label, sublabel, accent) in enumerate(row1):
    draw_card(ax, start_x1 + i * (card_w + gap), 2.9, card_w, card_h, number, label, sublabel, accent)

card_w2 = (total_w1 - gap) / 2
for i, (number, label, sublabel, accent) in enumerate(row2):
    draw_card(ax, start_x1 + i * (card_w2 + gap), 0.6, card_w2, card_h, number, label, sublabel, accent)

plt.tight_layout()
plt.savefig("summary_metrics.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


### 4.2 Word clouds — biggest challenges, split by ALICE status

In [ ]:
def uw_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    """Map word-cloud font sizes to the UWSM palette — biggest words get the boldest colours."""
    colors = [
        "#C8102E",  # UW red — biggest words
        "#E87722",  # UW amber
        "#003DA5",  # UW blue
        "#1B7A4A",  # deep green
        "#6B2D8B",  # purple
        "#0093B2",  # teal — smallest words
    ]
    return colors[int(font_size / 20) % len(colors)]


def build_challenge_freqs(df_subset):
    """Count canonical challenge labels in the cleaned biggest_challenge field. Drops 'None' and 'Other'."""
    labels = []
    for entry in df_subset["biggest_challenge_clean"].dropna():
        parts = [l.strip() for l in entry.split(";") if l.strip()]
        labels.extend([l for l in parts if l.lower() not in ("none", "other")])
    return Counter(labels)


def render_wordcloud(freq, title, filename):
    """Render a single word cloud using the UWSM colour function."""
    wc = WordCloud(
        width=1000, height=600,
        background_color=BG_COLOR,
        color_func=uw_color_func,
        max_words=40, prefer_horizontal=0.65,
        min_font_size=12, max_font_size=120,
        collocations=False, repeat=True,
    ).generate_from_frequencies(freq)

    fig, ax = plt.subplots(figsize=(12, 7))
    fig.patch.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
    ax.set_title(title, fontsize=15, fontweight="bold", color=LABEL_COLOR, pad=16)
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
    plt.show()


# Above ALICE
df_above = df_sm[df_sm["alice_clean"] == "Above ALICE"]
freq_above = build_challenge_freqs(df_above)
print(f"Above ALICE: n={len(df_above)}, top 5 challenges:")
for k, v in freq_above.most_common(5):
    print(f"  {k}: {v}")
render_wordcloud(freq_above,
                 "Biggest Challenges: Above ALICE Households",
                 "bc_above_alice_wordcloud_final.png")

# Below ALICE
df_below = df_sm[df_sm["alice_clean"] == "Below ALICE"]
freq_below = build_challenge_freqs(df_below)
print(f"\nBelow ALICE: n={len(df_below)}, top 5 challenges:")
for k, v in freq_below.most_common(5):
    print(f"  {k}: {v}")
render_wordcloud(freq_below,
                 "Biggest Challenges: Below ALICE Households",
                 "bc_below_alice_wordcloud_final.png")


### 4.3 Dumbbell chart — challenges by ALICE status

The headline figure of the project. Each row shows one challenge with two dots: red for Below ALICE, blue for Above ALICE. A gap between dots means the two groups cite that challenge at different rates. Note that housing cost (top row) shows almost no gap — the headline cross-class finding.

In [ ]:
# Citation rates computed by ALICE class. The hardcoded values below come from
# previous runs against the same dataset; running this cell against a fresh CSV
# will re-derive them, but the percentages are stable to the displayed precision.
challenges = [
    "Housing Cost", "Cost of Basics", "Mental Health", "Healthcare Cost",
    "Affordable Child Care", "Low Wages", "Food Access", "Behavioral Health",
    "Transportation", "Elder Care", "Social Isolation"
]
below = [68.8, 55.6, 34.8, 32.6, 21.6, 21.6, 20.6, 16.7, 14.1,  9.0,  0.0]
above = [65.2, 47.2, 34.5, 43.4, 33.9, 19.6, 11.2, 16.4,  0.0, 17.2, 13.3]

# Sort by Below value ascending — smallest at top is more readable
pairs = sorted(zip(challenges, below, above), key=lambda x: x[1])
challenges, below, above = zip(*pairs)

# ── FIGURE ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)

y_pos = range(len(challenges))

# Connecting line between dots emphasises the gap
for i, (b, a) in enumerate(zip(below, above)):
    ax.plot([b, a], [i, i], color="#D3D1C7", linewidth=2.5, zorder=1)

# Below ALICE: red circles. Above ALICE: blue diamonds.
ax.scatter(below, y_pos, color=RED_ACCENT,  s=120, zorder=3, label="Below ALICE")
ax.scatter(above, y_pos, color=BLUE_ACCENT, s=120, zorder=3, marker="D", label="Above ALICE")

# Value labels — only show where the gap is ≥ 5pp, otherwise the labels collide
for i, (b, a) in enumerate(zip(below, above)):
    if abs(b - a) < 5:
        continue
    ax.text(b, i - 0.28, f"{b:.0f}%", ha="center", va="top",    fontsize=8, color=RED_ACCENT)
    ax.text(a, i + 0.28, f"{a:.0f}%", ha="center", va="bottom", fontsize=8, color=BLUE_ACCENT)

ax.set_yticks(y_pos)
ax.set_yticklabels(challenges, fontsize=10, color=LABEL_COLOR)
ax.set_xlabel("% of Respondents Citing Challenge", fontsize=10, color=LABEL_COLOR)
ax.set_xlim(-10, 90)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))
ax.tick_params(axis="x", colors=SUB_COLOR, labelsize=9)
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True, color="#E0E0E0", linewidth=0.8, zorder=0)
ax.yaxis.grid(False)
for s in ax.spines.values():
    s.set_visible(False)

ax.set_title("Community Challenges: Below ALICE vs. Above ALICE Households",
             fontsize=14, fontweight="bold", color=LABEL_COLOR, pad=16)

leg = ax.legend(
    handles=[
        Line2D([0], [0], marker="o", color="w", markerfacecolor=RED_ACCENT,  markersize=9, label="Below ALICE"),
        Line2D([0], [0], marker="D", color="w", markerfacecolor=BLUE_ACCENT, markersize=9, label="Above ALICE"),
    ],
    loc="lower right", frameon=False, fontsize=10)
for t in leg.get_texts():
    t.set_color(LABEL_COLOR)

plt.tight_layout()
plt.savefig("bc_alice_nonalice_dumbbell.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


### 4.4 Challenges by age — slope chart and heatmap

The 25-64 age band is the peak-stress window across almost every challenge. Over-65 respondents continue to report high material hardship — they have not stabilized economically — and they pick up the elder-care concern in addition. Under-24 respondents report low rates of every challenge.

In [ ]:
# Per-age citation rates derived from the cleaned challenge indicators (see Section 3.4)
challenges  = ["Housing Cost", "Cost of Basics", "Healthcare Cost",
               "Mental Health", "Child Care", "Elder Care", "Low Wages"]
under24 = [12.5, 10.7,  7.1,  8.0,  0.0,  0.0, 4.5]
mid64   = [51.3, 39.3, 29.2, 28.6, 30.1,  0.0, 0.0]
over65  = [54.5, 38.4, 39.4,  0.0, 19.2, 21.2, 0.0]

line_colors = ["#003DA5", "#C8102E", "#E87722", "#1B7A4A", "#6B2D8B", "#0093B2", "#888780"]
age_labels  = ["Under 24", "25–64", "Over 65+"]
x = [0, 1, 2]

# ── SLOPE CHART ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)

for i, (challenge, color) in enumerate(zip(challenges, line_colors)):
    vals = [under24[i], mid64[i], over65[i]]
    ax.plot(x, vals, color=color, linewidth=2.2, marker="o", markersize=7, zorder=3, label=challenge)
    ax.text(2.05, over65[i], f"{challenge}  {over65[i]:.0f}%",
            va="center", ha="left", fontsize=8.5, color=color)

ax.set_xticks(x); ax.set_xticklabels(age_labels, fontsize=11, color=LABEL_COLOR)
ax.set_xlim(-0.3, 3.2); ax.set_ylim(-5, 75)
ax.set_ylabel("% of Respondents", fontsize=10, color=LABEL_COLOR)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v)}%"))
ax.tick_params(axis="y", colors=SUB_COLOR, labelsize=9)
ax.tick_params(axis="x", length=0)
ax.yaxis.grid(True, color="#E0E0E0", linewidth=0.8, zorder=0)
ax.xaxis.grid(False)
for s in ax.spines.values():
    s.set_visible(False)

ax.set_title("How Community Challenges Shift Across Age Groups",
             fontsize=14, fontweight="bold", color=LABEL_COLOR, pad=16)

plt.tight_layout()
plt.savefig("bc_age_slope.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


In [ ]:
# ── HEATMAP VIEW OF THE SAME DATA ─────────────────────────────────────────────
data = np.array([under24, mid64, over65])

fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)

# White-to-UW-blue gradient
cmap = LinearSegmentedColormap.from_list("uw", ["#F0F2F5", "#003DA5"])
im = ax.imshow(data, cmap=cmap, aspect="auto", vmin=0, vmax=70)

# Cell labels — white text on dark cells, dark text on light
for row in range(len(age_labels)):
    for col in range(len(challenges)):
        val = data[row, col]
        text_color = "white" if val > 35 else LABEL_COLOR
        ax.text(col, row, f"{val:.0f}%",
                ha="center", va="center",
                fontsize=10, fontweight="bold", color=text_color)

ax.set_xticks(range(len(challenges)))
ax.set_xticklabels(challenges, fontsize=10, color=LABEL_COLOR, rotation=20, ha="right")
ax.set_yticks(range(len(age_labels)))
ax.set_yticklabels(age_labels, fontsize=10, color=LABEL_COLOR)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)

cbar = fig.colorbar(im, ax=ax, orientation="vertical", pad=0.02, shrink=0.85)
cbar.set_label("% citing challenge", fontsize=9, color=SUB_COLOR)
cbar.ax.tick_params(labelsize=8, colors=SUB_COLOR)
cbar.outline.set_visible(False)

ax.set_title("Challenge Intensity by Age Group",
             fontsize=14, fontweight="bold", color=LABEL_COLOR, pad=16)

plt.tight_layout()
plt.savefig("bc_age_heatmap.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


### 4.5 Race comparison — directional only

The BIPOC subsample is n=30 after geographic filtering, which is too small for reliable significance testing. This figure is included in the appendix of the report to document the direction of the difference and to motivate oversampling in future rounds. Treat as directional only.

In [ ]:
challenges = ["Housing Cost", "Cost of Basics", "Healthcare Cost",
              "Affordable Child Care", "Mental Health", "Low Wages"]
white = [83.8, 64.6, 51.9, 47.2, 46.0,  0.0]
bipoc = [96.7, 63.3, 46.7, 36.7, 36.7,  0.0]

pairs = sorted(zip(challenges, white, bipoc), key=lambda x: x[1])
challenges, white, bipoc = zip(*pairs)

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)

y_pos = range(len(challenges))
for i, (w, b) in enumerate(zip(white, bipoc)):
    ax.plot([w, b], [i, i], color="#D3D1C7", linewidth=2.5, zorder=1)

ax.scatter(white, y_pos, color=BLUE_ACCENT, s=120, zorder=3, label="White")
ax.scatter(bipoc, y_pos, color=RED_ACCENT,  s=120, marker="D", zorder=3, label="BIPOC")

for i, (w, b) in enumerate(zip(white, bipoc)):
    if abs(w - b) < 5:
        continue
    ax.text(w, i - 0.28, f"{w:.0f}%", ha="center", va="top",    fontsize=8, color=BLUE_ACCENT)
    ax.text(b, i + 0.28, f"{b:.0f}%", ha="center", va="bottom", fontsize=8, color=RED_ACCENT)

ax.set_yticks(y_pos)
ax.set_yticklabels(challenges, fontsize=10, color=LABEL_COLOR)
ax.set_xlabel("% of Respondents Citing Challenge", fontsize=10, color=LABEL_COLOR)
ax.set_xlim(-5, 110)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))
ax.tick_params(axis="x", colors=SUB_COLOR, labelsize=9)
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True, color="#E0E0E0", linewidth=0.8, zorder=0)
ax.yaxis.grid(False)
for s in ax.spines.values():
    s.set_visible(False)

ax.set_title("Community Challenges: White vs. BIPOC Respondents",
             fontsize=14, fontweight="bold", color=LABEL_COLOR, pad=16)
ax.text(0.99, 0.01,
        "Note: White n=339, BIPOC n=30. BIPOC results directional only.",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=8, color=SUB_COLOR, style="italic")

leg = ax.legend(
    handles=[
        Line2D([0], [0], marker="o", color="w", markerfacecolor=BLUE_ACCENT, markersize=9, label="White (n=339)"),
        Line2D([0], [0], marker="D", color="w", markerfacecolor=RED_ACCENT,  markersize=9, label="BIPOC (n=30)"),
    ],
    loc="lower right", frameon=False, fontsize=10)
for t in leg.get_texts():
    t.set_color(LABEL_COLOR)

plt.tight_layout()
plt.savefig("bc_race_dumbbell.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


### 4.6 Feeling heard by ALICE status — donut comparison

In [ ]:
# Compute heard / unheard rates per ALICE class, restricted to respondents who answered the question
heard_data = {}
for group, subset in df_sm.groupby("alice_clean"):
    answered = subset[subset["feel_heard_clean"].notna()]
    total    = len(answered)
    unheard  = (answered["feel_heard_clean"] == "Unheard").sum()
    heard    = (answered["feel_heard_clean"] == "Heard").sum()
    heard_data[group] = {
        "Unheard": unheard / total * 100,
        "Heard":   heard   / total * 100,
        "n":       total,
    }

for group, vals in heard_data.items():
    print(f"{group} (n={vals['n']}): {vals['Unheard']:.1f}% unheard, {vals['Heard']:.1f}% heard")


In [ ]:
# Donut figure — uses the values printed above
groups  = ["Below ALICE", "Above ALICE"]
unheard = [heard_data["Below ALICE"]["Unheard"], heard_data["Above ALICE"]["Unheard"]]
heard   = [heard_data["Below ALICE"]["Heard"],   heard_data["Above ALICE"]["Heard"]]
ns      = [heard_data["Below ALICE"]["n"],       heard_data["Above ALICE"]["n"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
fig.patch.set_facecolor(BG_COLOR)
fig.text(0.5, 0.97, "Who Feels Heard? By ALICE Status",
         ha="center", va="top", fontsize=15, fontweight="bold", color=LABEL_COLOR)

for ax, group, u, h, n in zip(axes, groups, unheard, heard, ns):
    ax.set_facecolor(BG_COLOR)
    ax.pie([u, h], colors=[RED_ACCENT, BLUE_ACCENT],
           startangle=90, counterclock=False,
           wedgeprops=dict(width=0.45, edgecolor=BG_COLOR, linewidth=2))
    ax.text(0,  0.08,  f"{u:.0f}%", ha="center", va="center",
            fontsize=32, fontweight="bold", color=RED_ACCENT)
    ax.text(0, -0.22,  "feel unheard", ha="center", va="center",
            fontsize=11, color=LABEL_COLOR)
    ax.text(0, -0.72,  group, ha="center", va="center",
            fontsize=13, fontweight="bold", color=LABEL_COLOR)
    ax.text(0, -0.92,  f"n={n} respondents", ha="center", va="center",
            fontsize=9, color=SUB_COLOR, style="italic")
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.1, 1.1); ax.axis("off")

leg = fig.legend(
    handles=[mpatches.Patch(facecolor=RED_ACCENT, label="Unheard"),
             mpatches.Patch(facecolor=BLUE_ACCENT, label="Heard")],
    loc="lower center", ncol=2, frameon=False, fontsize=11,
    bbox_to_anchor=(0.5, 0.02))
for t in leg.get_texts():
    t.set_color(LABEL_COLOR)

plt.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.savefig("heard_alice.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


### 4.7 Engagement preferences — combined radar + bar chart

Among the 99 respondents who said yes to involvement, broken down by ALICE status. Volunteering and speaking up are roughly equally preferred across both groups. The divergence is sharp for donating (17% of Above-ALICE vs 0% of Below-ALICE, as would be expected) and sharing one's story (more common among Below-ALICE).

In [ ]:
# Filter to those who want to be involved, then compute engagement-type rates per ALICE class
df_involved = df_sm[df_sm["want_involvement_clean"] == "Yes"]
print(f"Total who want to get involved: {len(df_involved)}")

involve_cols = [c for c in df_sm.columns if c.startswith("how_involvement_")
                and "clean" not in c]

def involvement_rates(subset):
    n = len(subset)
    return {
        col.replace("how_involvement_", ""):
            pd.to_numeric(subset[col], errors="coerce").sum() / n * 100
        for col in involve_cols
    }

# Per ALICE class (Above n=58, Below n=40 based on previous runs)
for group, subset in df_involved.groupby("alice_clean"):
    print(f"\n{group} (n={len(subset)}):")
    for label, pct in involvement_rates(subset).items():
        print(f"  {label}: {pct:.1f}%")


In [ ]:
# Combined radar + bar layout — the figure as it appears in the deck
labels  = ["Volunteer", "Join Group", "Speak Up", "Share Story", "Donate", "Connect"]
above   = [58.6, 36.2, 39.7, 19.0, 17.2, 5.2]
below   = [55.0, 45.0, 40.0, 27.5,  0.0, 5.0]

above_closed = above + [above[0]]
below_closed = below + [below[0]]
n      = len(labels)
angles = [i * 2 * np.pi / n for i in range(n)] + [0]

fig = plt.figure(figsize=(14, 6))
fig.patch.set_facecolor(BG_COLOR)
ax_radar = fig.add_subplot(121, polar=True)
ax_bar   = fig.add_subplot(122)
fig.suptitle("How Do Community Members Want to Get Involved?",
             fontsize=14, fontweight="bold", color=LABEL_COLOR, y=1.01)

# ── RADAR ────────────────────────────────────────────────────────────────────
ax_radar.set_facecolor(BG_COLOR)
ax_radar.yaxis.grid(False); ax_radar.xaxis.grid(False)
for s in ax_radar.spines.values():
    s.set_visible(False)
for r in [20, 40, 60]:
    ax_radar.plot(angles, [r] * len(angles), color="#E0E0E0", linewidth=0.8, zorder=0)
for angle in angles[:-1]:
    ax_radar.plot([angle, angle], [0, 60], color="#E0E0E0", linewidth=0.8, zorder=0)

ax_radar.plot(angles, above_closed, color=BLUE_ACCENT, linewidth=2.5, zorder=3)
ax_radar.fill(angles, above_closed, color=BLUE_ACCENT, alpha=0.15, zorder=2)
ax_radar.plot(angles, below_closed, color=RED_ACCENT,  linewidth=2.5, zorder=3)
ax_radar.fill(angles, below_closed, color=RED_ACCENT,  alpha=0.15, zorder=2)
ax_radar.scatter(angles[:-1], above, color=BLUE_ACCENT, s=50, zorder=4)
ax_radar.scatter(angles[:-1], below, color=RED_ACCENT,  s=50, zorder=4)

ax_radar.set_theta_offset(np.pi / 2)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(labels, fontsize=9.5, color=LABEL_COLOR)
ax_radar.set_yticks([20, 40, 60])
ax_radar.set_yticklabels(["20%", "40%", "60%"], fontsize=7, color=SUB_COLOR)
ax_radar.set_ylim(0, 75)

# ── BAR ──────────────────────────────────────────────────────────────────────
ax_bar.set_facecolor(BG_COLOR)
order  = sorted(range(len(labels)), key=lambda i: above[i])
labels_s = [labels[i] for i in order]
above_s  = [above[i]  for i in order]
below_s  = [below[i]  for i in order]

y_pos = np.arange(len(labels)); bar_h = 0.35
bars_a = ax_bar.barh(y_pos + bar_h/2, above_s, bar_h, color=BLUE_ACCENT, alpha=0.9, zorder=2)
bars_b = ax_bar.barh(y_pos - bar_h/2, below_s, bar_h, color=RED_ACCENT,  alpha=0.9, zorder=2)

for bar, val in zip(bars_a, above_s):
    if val > 0:
        ax_bar.text(val + 1, bar.get_y() + bar.get_height()/2,
                    f"{val:.0f}%", va="center", fontsize=8.5, color=BLUE_ACCENT)
for bar, val in zip(bars_b, below_s):
    ax_bar.text(val + 1, bar.get_y() + bar.get_height()/2,
                f"{val:.0f}%" if val > 0 else "0%",
                va="center", fontsize=8.5, color=RED_ACCENT)

ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(labels_s, fontsize=10, color=LABEL_COLOR)
ax_bar.set_xlabel("% of Respondents", fontsize=10, color=LABEL_COLOR)
ax_bar.set_xlim(0, 75)
ax_bar.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v)}%"))
ax_bar.tick_params(axis="x", colors=SUB_COLOR, labelsize=9)
ax_bar.tick_params(axis="y", length=0)
ax_bar.xaxis.grid(True, color="#E0E0E0", linewidth=0.8, zorder=0)
ax_bar.yaxis.grid(False)
for s in ax_bar.spines.values():
    s.set_visible(False)

leg = fig.legend(
    handles=[
        Line2D([0], [0], color=BLUE_ACCENT, linewidth=2.5, marker="o", markersize=7, label="Above ALICE (n=58)"),
        Line2D([0], [0], color=RED_ACCENT,  linewidth=2.5, marker="o", markersize=7, label="Below ALICE (n=40)"),
    ],
    loc="lower center", ncol=2, frameon=False, fontsize=10,
    bbox_to_anchor=(0.5, -0.04))
for t in leg.get_texts():
    t.set_color(LABEL_COLOR)

fig.text(0.5, -0.08, "Among respondents who said yes to getting involved  |  Total n=99",
         ha="center", fontsize=8, color=SUB_COLOR, style="italic")

plt.tight_layout()
plt.savefig("engagement_combined_alice.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


### 4.8 Which communities feel unheard — donut

Built from the `community_super` field, which is the super-category collapse of the open-text community write-in (see `notebooks/01_cleaning_processing.ipynb` for the classification pipeline). `Other` is excluded so the chart reflects classifiable communities only.

In [ ]:
community_counts = df_sm["community_super"].value_counts().drop("Other", errors="ignore")
labels = community_counts.index.tolist()
sizes  = community_counts.values.tolist()
total  = sum(sizes)

colors = [
    "#003DA5",  # Town of Residence
    "#C8102E",  # BIPOC
    "#E87722",  # Women
    "#1B7A4A",  # Older Adults
    "#6B2D8B",  # LGBTQ+
    "#0093B2",  # Immigrant
    "#888780",  # Youth
    "#D4537E",  # Parents
    "#F7C1C1",  # Recovery Community
    "#B5D4F4",  # ALICE
    "#FAC775",  # Single Mothers
    "#C0DD97",  # Hispanic
    "#AFA9EC",  # White
][:len(labels)]

fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)

wedges, _, autotexts = ax.pie(
    sizes, labels=None, colors=colors,
    autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.55, edgecolor=BG_COLOR, linewidth=2),
    pctdistance=0.75,
)
for a in autotexts:
    a.set_fontsize(8.5); a.set_fontweight("bold"); a.set_color("white")

ax.text(0,  0.08, f"{total}", ha="center", va="center",
        fontsize=28, fontweight="bold", color=LABEL_COLOR)
ax.text(0, -0.18, "identified a\ncommunity", ha="center", va="center",
        fontsize=10, color=SUB_COLOR, linespacing=1.4)

legend_labels = [f"{l}  (n={s}, {s/total*100:.1f}%)" for l, s in zip(labels, sizes)]
patches = [mpatches.Patch(facecolor=c, label=l) for c, l in zip(colors, legend_labels)]
leg = ax.legend(handles=patches, loc="center left", bbox_to_anchor=(0.95, 0.5),
                frameon=False, fontsize=9.5)
for t in leg.get_texts():
    t.set_color(LABEL_COLOR)

ax.set_title("Which Communities Feel Unheard?",
             fontsize=14, fontweight="bold", color=LABEL_COLOR, pad=20)
fig.text(0.5, 0.02,
         "Note: 'Other' excluded (10.5%) — reflects genuinely unclassifiable free-text responses",
         ha="center", fontsize=8, color=SUB_COLOR, style="italic")

plt.tight_layout()
plt.savefig("specific_unheard_communities.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


## 5. Statistical validation — chi-square with Cramér's V

We run pairwise chi-square tests across three families of relationships:

1. ALICE status × each of the 24 binary challenge indicators
2. Feel heard × demographic variables (ALICE, race, age)
3. Want involvement × demographic and engagement variables

For each test we report n, p-value, significance stars, Cramér's V, and the conventional effect-size interpretation. With n ≈ 1,700, almost any non-trivial difference reaches significance, so the effect size is what matters substantively.


In [ ]:
def cramers_v(chi2, n, r, k):
    """Cramér's V effect size for a chi-square test."""
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))


def run_chi_square(df, col1, col2, label):
    """Run a chi-square test between two categorical columns, return a structured result."""
    subset = df[[col1, col2]].dropna()
    n      = len(subset)
    ct     = pd.crosstab(subset[col1], subset[col2])
    chi2, p, dof, expected = chi2_contingency(ct)
    r, k = ct.shape
    v = cramers_v(chi2, n, r, k)

    effect = "Weak" if v < 0.1 else ("Moderate" if v < 0.3 else "Strong")
    if   p < 0.001: sig = "***"
    elif p < 0.01:  sig = "**"
    elif p < 0.05:  sig = "*"
    else:           sig = "ns"

    return {
        "Comparison":  label, "N": n,
        "Chi2":        round(chi2, 2),
        "p-value":     round(p, 4),
        "Sig":         sig,
        "Cramér's V":  round(v, 3),
        "Effect":      effect,
    }


### 5.1 ALICE status × each individual challenge

In [ ]:
challenge_cols = [c for c in df_sm.columns
                  if c.startswith("biggest_challenge_")
                  and "None" not in c
                  and "Other" not in c
                  and c != "biggest_challenge_clean"]

alice_results = []
for col in challenge_cols:
    label = col.replace("biggest_challenge_", "").replace("_", " ")
    # Convert the one-hot column to a string label so chi-square sees two categories
    df_sm[f"{col}_str"] = pd.to_numeric(df_sm[col], errors="coerce").map({0: "Not Selected", 1: "Selected"})
    alice_results.append(run_chi_square(df_sm, "alice_clean", f"{col}_str", label))
    df_sm.drop(columns=[f"{col}_str"], inplace=True)

alice_df = pd.DataFrame(alice_results).sort_values("Cramér's V", ascending=False)
print(alice_df[["Comparison", "N", "p-value", "Sig", "Cramér's V", "Effect"]].to_string(index=False))


### 5.2 Feel heard × demographics; want involvement × demographics

In [ ]:
heard_tests = []
for col, label in [("alice_clean",         "Feel Heard vs ALICE Status"),
                   ("race_ethnicity_clean","Feel Heard vs Race/Ethnicity"),
                   ("age_group_clean",     "Feel Heard vs Age Group")]:
    heard_tests.append(run_chi_square(df_sm, "feel_heard_clean", col, label))

involve_tests = []
for col, label in [("alice_clean",         "Want Involvement vs ALICE Status"),
                   ("feel_heard_clean",    "Want Involvement vs Feel Heard"),
                   ("age_group_clean",     "Want Involvement vs Age Group"),
                   ("race_ethnicity_clean","Want Involvement vs Race/Ethnicity")]:
    involve_tests.append(run_chi_square(df_sm, "want_involvement_clean", col, label))

print("── Feel heard × demographics ──")
print(pd.DataFrame(heard_tests)[["Comparison", "N", "p-value", "Sig", "Cramér's V", "Effect"]].to_string(index=False))
print("\n── Want involvement × variables ──")
print(pd.DataFrame(involve_tests)[["Comparison", "N", "p-value", "Sig", "Cramér's V", "Effect"]].to_string(index=False))


### 5.3 Summary visualization — one figure, three panels

In [ ]:
# Statistically significant relationships, grouped by theme
sections = {
    "Challenges vs ALICE Status\n(top significant only)": [
        ("Food Access",                0.122, "***", "Moderate"),
        ("Affordable Child Care",      0.121, "***", "Moderate"),
        ("Affordable Elder Care",      0.104, "***", "Moderate"),
        ("Healthcare Cost",            0.100, "***", "Weak"),
        ("Cost of Basics",             0.075, "**",  "Weak"),
        ("Social Isolation",           0.072, "**",  "Weak"),
        ("Transportation",             0.065, "**",  "Weak"),
        ("Education Access",           0.054, "*",   "Weak"),
    ],
    "Feeling Heard vs Demographics": [
        ("ALICE Status",               0.205, "***", "Moderate"),
        ("Race / Ethnicity",           0.214, "**",  "Moderate"),
        ("Age Group",                  0.072, "ns",  "Weak"),
    ],
    "Want Involvement vs Demographics": [
        ("Race / Ethnicity",           0.224, "***", "Moderate"),
        ("ALICE Status",               0.172, "***", "Moderate"),
        ("Feel Heard",                 0.061, "ns",  "Weak"),
        ("Age Group",                  0.082, "ns",  "Weak"),
    ],
}

def bar_color(sig, effect):
    """Colour bars by significance + effect-size tier."""
    if sig == "ns":           return "#D3D1C7"      # gray — not significant
    if effect == "Moderate":  return RED_ACCENT     # red — sig + moderate
    return AMBER_ACCENT                              # amber — sig + weak

fig, axes = plt.subplots(1, 3, figsize=(16, 7))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle("Statistical Validation: Key Relationships (Chi-Square + Cramér's V)",
             fontsize=14, fontweight="bold", color=LABEL_COLOR, y=1.01)

for ax, (section_title, items) in zip(axes, sections.items()):
    ax.set_facecolor(BG_COLOR)
    labels  = [item[0] for item in items]
    values  = [item[1] for item in items]
    sigs    = [item[2] for item in items]
    effects = [item[3] for item in items]
    colors  = [bar_color(s, e) for s, e in zip(sigs, effects)]

    y_pos = np.arange(len(labels))
    bars  = ax.barh(y_pos, values, color=colors, alpha=0.92, zorder=2)

    for bar, val, sig in zip(bars, values, sigs):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
                f"{val:.3f} {sig}", va="center", fontsize=8.5,
                color=LABEL_COLOR if sig != "ns" else SUB_COLOR)

    ax.axvline(x=0.1, color=LABEL_COLOR, linewidth=1, linestyle="--", alpha=0.3, zorder=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9, color=LABEL_COLOR)
    ax.set_xlabel("Cramér's V", fontsize=9, color=LABEL_COLOR)
    ax.set_title(section_title, fontsize=10, fontweight="bold", color=LABEL_COLOR, pad=10)
    ax.set_xlim(0, 0.32)
    ax.tick_params(axis="x", colors=SUB_COLOR, labelsize=8)
    ax.tick_params(axis="y", length=0)
    ax.xaxis.grid(True, color="#E0E0E0", linewidth=0.8, zorder=0)
    ax.yaxis.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

leg = fig.legend(
    handles=[
        mpatches.Patch(facecolor=RED_ACCENT,   label="Significant + Moderate effect"),
        mpatches.Patch(facecolor=AMBER_ACCENT, label="Significant + Weak effect"),
        mpatches.Patch(facecolor="#D3D1C7",    label="Not significant (shown for context)"),
    ],
    loc="lower center", ncol=3, frameon=False, fontsize=9,
    bbox_to_anchor=(0.5, -0.04))
for t in leg.get_texts():
    t.set_color(LABEL_COLOR)

fig.text(0.5, -0.08,
         "* p<0.05  ** p<0.01  *** p<0.001  ns = not significant  |  Cramér's V: moderate ≥ 0.1",
         ha="center", fontsize=8, color=SUB_COLOR, style="italic")

plt.tight_layout()
plt.savefig("slide_chi_square_summary.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


## 6. Predictive modeling — can we identify Below-ALICE status from challenge profiles?

We fit two classifiers — a class-balanced logistic regression and a depth-4 decision tree — to predict Below-ALICE status from the 24 binary challenge indicators. Both are evaluated under stratified 5-fold cross-validation, and both significantly underperform a trivial majority-class baseline (0.70 accuracy). That's not a failure; it's a finding. The challenges a respondent reports do not, on their own, cleanly identify ALICE status. See [`docs/methods.md`](../docs/methods.md) for the construct-validation framing of why we ran these models anyway.


### 6.1 Feature set

In [ ]:
df_model = df_sm[df_sm["alice_clean"].notna()].copy()
df_model["target"] = (df_model["alice_clean"] == "Below ALICE").astype(int)

print(f"Total samples:    {len(df_model)}")
print(f"Below ALICE (1):  {df_model['target'].sum()} ({df_model['target'].mean()*100:.1f}%)")
print(f"Above ALICE (0):  {(df_model['target']==0).sum()} ({(df_model['target']==0).mean()*100:.1f}%)")

# Drop None and Other from the feature set — they're not real challenges
challenge_cols = [c for c in df_sm.columns
                  if c.startswith("biggest_challenge_")
                  and "None" not in c
                  and "Other" not in c
                  and c != "biggest_challenge_clean"]

X = df_model[challenge_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y = df_model["target"]
feature_names = [c.replace("biggest_challenge_", "").replace("_", " ") for c in challenge_cols]
print(f"\nFeatures: {len(feature_names)}")


### 6.2 Logistic regression — odds ratios

In [ ]:
clf_lr = LogisticRegression(
    class_weight="balanced",     # corrects the 30/70 class imbalance
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver="lbfgs",
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_validate(
    clf_lr, X, y, cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    return_train_score=False,
)

print("── Logistic Regression: stratified 5-fold CV ──")
print(f"  Accuracy:  {scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}")
print(f"  Precision: {scores['test_precision'].mean():.3f} ± {scores['test_precision'].std():.3f}")
print(f"  Recall:    {scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}")
print(f"  F1 Score:  {scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}")


In [ ]:
# Final fit on full sample for coefficient interpretation
clf_lr.fit(X, y)
coefficients = clf_lr.coef_[0]
odds_ratios  = np.exp(coefficients)

coef_df = pd.DataFrame({
    "Challenge":   feature_names,
    "Coefficient": coefficients,
    "Odds Ratio":  odds_ratios,
}).sort_values("Odds Ratio", ascending=False)

print("── Odds ratios — sorted ──")
print("OR > 1: citing this challenge increases the odds of being Below ALICE")
print("OR < 1: citing this challenge decreases the odds of being Below ALICE\n")
for _, row in coef_df.iterrows():
    direction = "↑ Below ALICE" if row["Odds Ratio"] > 1 else "↓ Above ALICE"
    print(f"  {row['Challenge']:<35} OR={row['Odds Ratio']:.2f}  {direction}")


In [ ]:
# Visualize odds ratios — show only meaningful effects (OR > 1.1 or OR < 0.9)
plot_df = coef_df[(coef_df["Odds Ratio"] > 1.1) | (coef_df["Odds Ratio"] < 0.9)].copy()
plot_df = plot_df.sort_values("Odds Ratio")

colors = [RED_ACCENT if or_ > 1 else BLUE_ACCENT for or_ in plot_df["Odds Ratio"]]

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)

bars = ax.barh(plot_df["Challenge"], plot_df["Odds Ratio"],
               color=colors, alpha=0.9, zorder=2)

ax.axvline(x=1, color=LABEL_COLOR, linewidth=1.2, linestyle="--", zorder=3, alpha=0.6)
ax.text(1.01, ax.get_ylim()[1] * 0.98, "No effect", fontsize=8, color=SUB_COLOR, va="top")

for bar, val in zip(bars, plot_df["Odds Ratio"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}x", va="center", fontsize=9, color=LABEL_COLOR)

ax.set_xlabel("Odds Ratio", fontsize=10, color=LABEL_COLOR)
ax.set_title("Which Challenges Predict ALICE Status?\n(Logistic Regression Odds Ratios)",
             fontsize=13, fontweight="bold", color=LABEL_COLOR, pad=16)
ax.tick_params(axis="y", length=0, labelsize=10)
ax.tick_params(axis="x", colors=SUB_COLOR, labelsize=9)
ax.xaxis.grid(True, color="#E0E0E0", linewidth=0.8, zorder=0)
ax.yaxis.grid(False)
for s in ax.spines.values():
    s.set_visible(False)

leg = ax.legend(
    handles=[Patch(facecolor=RED_ACCENT,  label="Increases odds of Below ALICE"),
             Patch(facecolor=BLUE_ACCENT, label="Decreases odds of Below ALICE")],
    frameon=False, fontsize=9, loc="lower right")
for t in leg.get_texts():
    t.set_color(LABEL_COLOR)

fig.text(0.5, 0.01,
         f"Only challenges with OR > 1.1 or OR < 0.9 shown  |  All survey types  |  n={len(X)}",
         ha="center", fontsize=8, color=SUB_COLOR, style="italic")

plt.tight_layout()
plt.savefig("lr_alice_odds_ratios.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


### 6.3 Decision tree — feature importance

In [ ]:
def run_cv_tree(X, y, max_depth):
    """Stratified 5-fold CV for a depth-limited decision tree."""
    clf = DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=max_depth,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_validate(clf, X, y, cv=cv,
                            scoring=["accuracy", "precision", "recall", "f1"])
    return scores

print("── Decision Tree CV across depths ──")
for d in (3, 4, 5):
    s = run_cv_tree(X, y, d)
    print(f"  depth={d}  acc={s['test_accuracy'].mean():.3f}  "
          f"prec={s['test_precision'].mean():.3f}  "
          f"rec={s['test_recall'].mean():.3f}  "
          f"f1={s['test_f1'].mean():.3f}")


In [ ]:
# Fit final tree at depth 4 and inspect feature importances
clf_dt = DecisionTreeClassifier(
    class_weight="balanced",
    max_depth=4,
    min_samples_leaf=10,
    random_state=RANDOM_STATE,
)
clf_dt.fit(X, y)

importances = clf_dt.feature_importances_
indices     = np.argsort(importances)[::-1]

print("── Top 10 features by importance ──")
for i in range(min(10, len(feature_names))):
    print(f"  {i+1:2d}. {feature_names[indices[i]]:<35} {importances[indices[i]]:.3f}")

# Plot top 10
top_n   = 10
top_idx = indices[:top_n]
top_names = [feature_names[i] for i in top_idx]
top_imps  = importances[top_idx]

bar_colors = [RED_ACCENT  if imp > 0.10 else
              BLUE_ACCENT if imp > 0.05 else
              AMBER_ACCENT for imp in top_imps]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)

bars = ax.barh(range(top_n), top_imps[::-1], color=bar_colors[::-1], alpha=0.9, zorder=2)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_names[::-1], fontsize=10, color=LABEL_COLOR)
ax.set_xlabel("Feature Importance", fontsize=10, color=LABEL_COLOR)
ax.set_title("Which Challenges Best Predict ALICE Status?",
             fontsize=14, fontweight="bold", color=LABEL_COLOR, pad=16)

for bar, val in zip(bars, top_imps[::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9, color=LABEL_COLOR)

ax.set_xlim(0, max(top_imps) + 0.05)
ax.tick_params(axis="x", colors=SUB_COLOR, labelsize=9)
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True, color="#E0E0E0", linewidth=0.8, zorder=0)
ax.yaxis.grid(False)
for s in ax.spines.values():
    s.set_visible(False)

leg = ax.legend(
    handles=[Patch(facecolor=RED_ACCENT,   label="High importance (>0.10)"),
             Patch(facecolor=BLUE_ACCENT,  label="Medium importance (0.05-0.10)"),
             Patch(facecolor=AMBER_ACCENT, label="Lower importance (<0.05)")],
    frameon=False, fontsize=9, loc="lower right")
for t in leg.get_texts():
    t.set_color(LABEL_COLOR)

plt.tight_layout()
plt.savefig("dt_alice_feature_importance.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


## 7. PCA — dimensionality reduction on the challenge matrix

We project the 24-feature challenge matrix into a 2-dimensional PCA space to test whether challenge profiles can be summarized by a small number of underlying axes. The answer is largely no — the first two components together explain only ~16% of variance — but the loadings are still interpretable: PC1 reads as "severity of everyday material hardship" and PC2 as "systemic / marginal issue." Neither separates Below-ALICE from Above-ALICE on its own, which is consistent with the modest classifier performance above.

In [ ]:
X_for_pca = df_model[challenge_cols].apply(pd.to_numeric, errors="coerce").fillna(0).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_for_pca)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print("── Variance explained ──")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"  Total: {sum(pca.explained_variance_ratio_)*100:.1f}%")

loadings = pd.DataFrame(pca.components_.T, index=feature_names, columns=["PC1", "PC2"])
print("\nTop 5 challenges driving PC1 (severity-of-everyday-hardship axis):")
print(loadings["PC1"].abs().sort_values(ascending=False).head(5))
print("\nTop 5 challenges driving PC2 (systemic/marginal-issue axis):")
print(loadings["PC2"].abs().sort_values(ascending=False).head(5))


In [ ]:
# 2D scatter coloured by ALICE class
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(BG_COLOR); ax.set_facecolor(BG_COLOR)

for group, color in {"Below ALICE": RED_ACCENT, "Above ALICE": BLUE_ACCENT}.items():
    mask = df_model["alice_clean"] == group
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=group, alpha=0.4, s=20, zorder=2)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=10, color=LABEL_COLOR)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=10, color=LABEL_COLOR)
ax.set_title("PCA of Community Challenges\nColored by ALICE Status",
             fontsize=14, fontweight="bold", color=LABEL_COLOR, pad=16)
ax.tick_params(colors=SUB_COLOR, labelsize=9)
ax.xaxis.grid(True, color="#E0E0E0", linewidth=0.8, zorder=0)
ax.yaxis.grid(True, color="#E0E0E0", linewidth=0.8, zorder=0)
for s in ax.spines.values():
    s.set_visible(False)

leg = ax.legend(frameon=False, fontsize=10, loc="upper right")
for t in leg.get_texts():
    t.set_color(LABEL_COLOR)

fig.text(0.5, 0.01,
         f"PCA on {X_for_pca.shape[1]} challenge columns  |  n={X_for_pca.shape[0]}  |  Total variance: {sum(pca.explained_variance_ratio_)*100:.1f}%",
         ha="center", fontsize=8, color=SUB_COLOR, style="italic")

plt.tight_layout()
plt.savefig("pca_alice_basic.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


## 8. Wrap-up

Every figure that appears in the final report and the community-facing deck has now been rendered from the processed dataset. For interpretation, the chain of reasoning that ties these findings together, and the full set of programming recommendations to UWSM, see [`report/final_report.pdf`](../report/final_report.pdf).

A few methodological footnotes that are easy to miss in the report and worth flagging here:

- Every subgroup analysis is restricted to the survey types that asked the relevant question. The Abbreviated survey (58% of respondents) only asked three questions; respondents who took it are absent from any analysis that needs age, race, feel-heard, or engagement.
- The BIPOC subsample is n=30 after geographic filtering. Race/ethnicity findings in Section 4.5 and 5.2 are directional only.
- The K-means segmentation reported in the deck and report uses a fixed random seed. Different seeds produce meaningfully different cluster assignments. Treat the four-cluster solution as exploratory, not as a stable taxonomy.
- The logistic regression in Section 6 was fit for construct validation, not for deployment. With F1 = 0.466 against a 30/70 split, it doesn't reliably identify ALICE status; the value of the model is that the *features that matter* match what poverty research independently identifies as markers of material deprivation.